# Fase 2 — Limpieza semántica y normalización a Markdown (Gemini)

Toma la salida de la Fase 1 (`8_final_cleaned.json`) y convierte el texto, todavía con saltos de línea artificiales y sin estructura, en Markdown limpio y bien formateado, usando los modelos `gemini-2.5-flash` / `gemini-2.5-pro` como motor de reformateo.

El proceso, encapsulado en `MedicalProtocolProcessor`, sigue estos pasos por documento:

1. **Limpieza mecánica** (`clean_text_surface`, `repair_broken_content`): normalización Unicode, unificación de viñetas, reparación de palabras y URLs partidas por saltos de línea, y eliminación de referencias numéricas intratexto.
2. **Troceado inteligente** (`split_text_smartly`): divide el documento en fragmentos de ~5750 caracteres, cortando preferiblemente en un punto y aparte para no romper frases a mitad.
3. **Limpieza con Gemini** (`call_gemini_cleaning`): envía cada fragmento al modelo con instrucciones estrictas de fidelidad (prohibido resumir, solo reformatear a Markdown), manteniendo continuidad con el fragmento anterior.
4. **Comprobaciones de seguridad**: tras cada respuesta se valida que no se haya perdido contenido mediante tres chequeos —longitud (`safety_length_check`), tokens clínicos críticos como dosis o fechas (`safety_semantic_check`) y similitud estructural tipo diff (`safety_structural_check`)—. Si alguna falla, se reintenta hasta 5 veces escalando a un modelo más potente y a un prompt de "rescate".

El resultado es `corpus_rag_final_gemini.json` (corpus limpio en Markdown, listo para el chunking de la Fase 3), junto con una versión en texto plano para revisión visual y dos reportes de incidencias (redundancias detectadas y fallos críticos que no se pudieron resolver automáticamente, para revisión manual). La última celda calcula estadísticas de granularidad del corpus resultante.

> **Nota de seguridad:** la clave de la API de Gemini se ha sustituido por un marcador de posición (`TU_GOOGLE_API_KEY_AQUI`); debe configurarse como variable de entorno o sustituirse por una clave propia antes de ejecutar el notebook.

In [ ]:
pip install --upgrade google-generativeai

In [ ]:
import google.generativeai as genai
import os

GOOGLE_API_KEY = "TU_GOOGLE_API_KEY_AQUI"
genai.configure(api_key=GOOGLE_API_KEY)

print("Listando modelos disponibles...")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f"- {m.name}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Listando modelos disponibles...
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-3-1b-it
- models/gemma-3-4b-it
- models/gemma-3-12b-it
- models/gemma-3-27b-it
- models/gemma-3n-e4b-it
- models/gemma-3n-e2b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-2.5-flash-lite-preview-09-2025
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3-pro-image-preview
- models/nano-banana-pro-preview
- models/gemini-robotics-er-1.5-preview
- models/gemini-2.5-computer-use-preview-10-2025
- models/deep-research-pro-preview-12-2025


In [ ]:
import json
import os
import re
import unicodedata
import time
import google.generativeai as genai
from tqdm import tqdm  # Barra de progreso
import random
from collections import Counter
import difflib

# --- CONFIGURACIÓN DE GEMINI ---
GOOGLE_API_KEY = "TU_GOOGLE_API_KEY_AQUI" # O usa os.getenv('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

model_flash = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    generation_config={
        "temperature": 0.0,  # CRÍTICO: Determinismo total
        "top_p": 1,
        "max_output_tokens": 8192,
    },
    system_instruction="""
    Eres un asistente experto en digitalización de documentos médicos.
    Recibirás un texto que ya ha sido pre-procesado, pero que aún tiene "saltos de línea artificiales" dentro de los párrafos y le falta estructura.

    OBJETIVO: Convertir texto médico desordenado (OCR) a Markdown limpio.

    DIRECTRICES DE SEGURIDAD ABSOLUTA:
    1. PROHIBIDO RESUMIR: Debes mantener CADA palabra, dato, cifra y nombre del original.
    2. VERBATIM: El contenido informativo debe ser idéntico al original palabra por palabra.
    3. FORMATO: Solo arregla saltos de línea (\n) rotos y aplica estructura Markdown (#, -, |).
    4. Detecta Títulos y Subtítulos: Usa SIEMPRE "## " (doble almohadilla) para CUALQUIER encabezado detectado.
    5. Si detectas datos tabulares, formatéalos como tablas Markdown (| Col | Col |). Si la tabla está muy rota, conviértela en una lista de viñetas clara.
    6. NO repitas el texto que se te da como 'CONTEXTO ANTERIOR'. Solo procesa el 'TEXTO A LIMPIAR'.

    SI EL TEXTO ES ILEGIBLE O RUIDO, DEVUÉLVELO TAL CUAL. NO LO ELIMINES.
    """
)

model_pro = genai.GenerativeModel(
    model_name="gemini-2.5-pro",
    generation_config={
        "temperature": 0.0,  # CRÍTICO: Determinismo total
        "top_p": 1,
        "max_output_tokens": 8192,
    },
    system_instruction="""
    Eres un asistente experto en digitalización de documentos médicos.
    Recibirás un texto que ya ha sido pre-procesado, pero que aún tiene "saltos de línea artificiales" dentro de los párrafos y le falta estructura.

    OBJETIVO: Convertir texto médico desordenado (OCR) a Markdown limpio.

    DIRECTRICES DE SEGURIDAD ABSOLUTA:
    1. PROHIBIDO RESUMIR: Debes mantener CADA palabra, dato, cifra y nombre del original.
    2. VERBATIM: El contenido informativo debe ser idéntico al original palabra por palabra.
    3. FORMATO: Solo arregla saltos de línea (\n) rotos y aplica estructura Markdown (#, -, |).
    4. Detecta Títulos y Subtítulos: Usa SIEMPRE "## " (doble almohadilla) para CUALQUIER encabezado detectado.
    5. Si detectas datos tabulares, formatéalos como tablas Markdown (| Col | Col |). Si la tabla está muy rota, conviértela en una lista de viñetas clara.
    6. NO repitas el texto que se te da como 'CONTEXTO ANTERIOR'. Solo procesa el 'TEXTO A LIMPIAR'.

    SI EL TEXTO ES ILEGIBLE O RUIDO, DEVUÉLVELO TAL CUAL. NO LO ELIMINES.
    """
)

# --- 2. CLASE DE LIMPIEZA MECÁNICA ---
class MedicalProtocolProcessor:
    """
    Limpia y normaliza el texto extraído de los PDF (Fase 1) y lo convierte
    en Markdown estructurado mediante el modelo Gemini. Aplica primero una
    limpieza mecánica (Unicode, viñetas, palabras/URLs partidas por saltos
    de línea, referencias intratexto) y después una serie de comprobaciones
    de seguridad (longitud, tokens clínicos críticos, similitud estructural)
    que detectan si el modelo ha alterado o perdido información, forzando
    reintentos con un modelo más potente cuando es necesario.
    """
    def __init__(self):
        """
        Inicializa el mapa de normalización de viñetas, el control de frecuencia
        de llamadas a la API y los registros (logs) de incidencias y errores
        críticos detectados durante el procesamiento.
        """
        # Mapeo exhaustivo de viñetas
        self.bullet_map = {
            '': '-', '': '-', '': '-', '•': '-',
            '\uf0b7': '-', '\u2013': '-', '➤': '-', '➢': '-', '■': '-'
        }
        self.last_call_time = 0
        self.min_interval = 0.5  # ← clave
        # --- NUEVO: Lista para guardar las incidencias para revisión manual ---
        self.redundancy_log = []
        self.error_log = []       # Fallos críticos (texto crudo devuelto)
        self.error_log_json = []


    def global_throttle(self):
        """
        Espera el tiempo necesario para no superar la frecuencia mínima entre
        llamadas a la API de Gemini (rate limiting).
        """
        now = time.time()
        elapsed = now - self.last_call_time

        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)

        self.last_call_time = time.time()

    def clean_text_surface(self, text):
        """Limpieza superficial de caracteres y artefactos."""
        if not text: return ""

        # 1. Normalización Unicode (NFKC)
        text = unicodedata.normalize('NFKC', text)

        # --- NUEVO: PEGAMENTO DE UNIDADES HUÉRFANAS ---
        # Si encuentra "17.6" seguido de salto de línea y luego "g" o "mg", lo junta.
        # Convierte "17.6 \n g" en "17.6 g" para que Gemini no se confunda.
        units_pattern = r'(mg|µg|mcg|gr?|kg|ml|l|mmhg|mmol/l|ui|u/l|%)'
        # Busca: Numero + salto linea + unidad
        text = re.sub(r'(\d+(?:[.,]\d+)?)\s*\n\s*(' + units_pattern + r')\b', r'\1 \2', text, flags=re.IGNORECASE)

        # 4. Estandarizar viñetas usando el mapa
        for char, replacement in self.bullet_map.items():
            text = text.replace(char, replacement)

        # 2. Eliminar caracteres de control no imprimibles (excepto saltos de línea)
        text = "".join(ch for ch in text if ch == "\n" or unicodedata.category(ch)[0] != "C")

        # 3. Eliminar Soft Hyphens (guiones invisibles)
        text = text.replace('\u00ad', '')


        # 5. Estandarizar viñetas que quedaron como saltos de línea raros
        bullet_pattern = r'(\n\s*)([-])(\s+)'
        text = re.sub(bullet_pattern, r'\n- ', text)

        return text

    def repair_broken_content(self, text):
        """Reparación de contenido roto (URLs, palabras, tablas)."""

        # 1. Reparar URLs rotas
        text = re.sub(r"(https?://\S+)\s+(\S+)", r"\1\2", text)

        # 2a. REGLA DE PALABRAS (De-hyphenation estándar)
        # Si son letras, quitamos el guion y unimos.
        # Ej: "infec-\nción" -> "infección"
        text = re.sub(r'([a-zA-ZáéíóúüñÁÉÍÓÚÜÑ]+)-\s*\n\s*([a-zA-ZáéíóúüñÁÉÍÓÚÜÑ]+)', r'\1\2', text)

        # 2b. REGLA DE RANGOS NUMÉRICOS (¡NUEVO!)
        # Si son números, quitamos el salto de línea pero DEJAMOS el guion.
        # Ej: "500-\n750" -> "500-750" (NO "500750")
        text = re.sub(r'(\d+)-\s*\n\s*(\d+)', r'\1-\2', text)

        # 3. Limpieza AVANZADA de referencias bibliográficas
        # Lógica mejorada:
        # - Lookbehind: Busca minúscula ([a-z]) O comillas (["'”’]).
        # - Match: Busca 1-3 dígitos seguidos opcionalmente de bloques ",dígitos".
        # - Lookahead: Debe terminar en espacio, punto, coma o fin de línea.
        # Esto elimina "texto1", "texto1,2,3", "“cita”4" pero respeta "H1N1" y "plan2024".

        ref_pattern = r'(?<=[a-z"\'”’\)])(\d{1,3}(?:,\d{1,3})*)(?=\s|[.,;]|$)'
        text = re.sub(ref_pattern, '', text)

        return text

    # --- AQUÍ ESTÁ LA MEJORA QUE HAS PEDIDO ---
    def split_text_smartly(self, text, chunk_size=6000):
        """
        Divide el texto buscando preferiblemente un punto y aparte (. + \n).
        """
        chunks = []
        start = 0
        text_len = len(text)

        while start < text_len:
            end = min(start + chunk_size, text_len)

            if end < text_len:
                # Miramos los últimos 1000 caracteres del bloque para buscar un buen punto de corte
                search_zone_start = max(start, end - 1000)
                search_zone = text[search_zone_start:end]

                # 1. PRIORIDAD ALTA: Buscar . ! ? seguido de salto de línea
                match = None
                # finditer encuentra todos, nos quedamos con el último (el más cercano al final)
                for m in re.finditer(r'[.!?]\s*\n', search_zone):
                    match = m

                if match:
                    # Cortamos justo después del salto de línea encontrado
                    end = search_zone_start + match.end()
                else:
                    # 2. PRIORIDAD BAJA: Si no hay puntos, buscamos cualquier salto de línea (Fallback)
                    last_newline = text.rfind('\n', start, end)
                    if last_newline != -1:
                        end = last_newline + 1

            chunk = text[start:end]
            chunks.append(chunk)
            start = end

        return chunks

    # --- FUNCIÓN DE SEGURIDAD (La clave para que no corte texto) ---
    def safety_length_check(self, original, cleaned):
        """
        Comprueba que el texto limpiado por Gemini no se haya reducido más de
        lo razonable respecto al original (umbral del 85%), lo que indicaría
        pérdida de información.
        """
        len_orig = len(original)
        len_clean = len(cleaned)

        if len_orig == 0: return True, "OK"

        # El texto limpio suele ser un poco más largo por el markdown.
        # Si es menor del 85% del original, es sospechoso.
        ratio = len_clean / len_orig

        if ratio < 0.85:
            return False, f"ALERTA: El texto se ha reducido demasiado ({ratio:.2%}). Posible pérdida de info."
        return True, "OK"


    def extract_critical_tokens(self, text):
        """
        Extrae del texto los tokens clínicos críticos (dosis y unidades,
        porcentajes, temperaturas, fechas y años) que deben conservarse intactos
        tras la limpieza, normalizando su formato para poder comparar el texto
        original y el limpiado de forma robusta.
        """
        tokens = []
        # Normalizamos para que no importen mayúsculas/minúsculas en el conteo general,
        # PERO detectamos acrónimos antes de bajar a lower() si quisiéramos ser estrictos.
        # Aquí usaremos lower para robustez general.
        t = text.lower()

        def normalize_token(match_list):
            """
            Normaliza los tokens extraídos (coma decimal -> punto, sin espacios)
            para que variaciones de formato no se cuenten como tokens distintos.
            """
            # 1. Reemplazar coma decimal por punto
            # 2. Eliminar TODOS los espacios en blanco (así "5 mg" == "5mg")
            return [m.replace(',',('.')).replace(' ', '').strip() for m in match_list]

        # 1. Valores clínicos y unidades
        raw_vals = re.findall(r'\b\d+(?:[\.,]\d+)?\s*(?:mg|µg|mcg|gr?|kg|ml|l|mmhg|mmol/l|ui|u/l)\b', t)
        tokens.extend(normalize_token(raw_vals)) # <--- AQUI APLICAMOS LA NORMALIZACIÓN

        # 2. Porcentajes y temperaturas
        raw_pct = re.findall(r'\b\d+(?:[\.,]\d+)?\s*(?:%|ºc|°c)\b', t)
        tokens.extend(normalize_token(raw_pct)) # <--- AQUI TAMBIÉN

        # 3. Fechas y Años
        tokens.extend(re.findall(r'\b\d{1,2}/\d{1,2}/\d{2,4}\b', t))
        tokens.extend(re.findall(r'\b(?:19|20)\d{2}\b', t))

        return tokens

    def safety_semantic_check(self, original, cleaned):
        """
        Compara los tokens críticos (extract_critical_tokens) del texto original
        y del limpiado: si algún dato desaparece por completo se considera un
        fallo crítico, y si solo se reduce su frecuencia se registra como
        advertencia de redundancia (pero no bloquea el resultado).
        """
        orig_counts = Counter(self.extract_critical_tokens(original))
        clean_counts = Counter(self.extract_critical_tokens(cleaned))
        missing = []
        redundancy_warnings = [] # Lista local de advertencias

        for token, count in orig_counts.items():
            clean_c = clean_counts[token]

            # 1. DATO DESAPARECIDO (Fallo Crítico)
            if clean_c == 0:
                missing.append(f"{token} (DESAPARECIDO)")

            # 2. REDUCCIÓN DE REDUNDANCIA (Advertencia, pero permitido)
            elif clean_c < count:
                msg = f"{token} (Input: {count} -> Output: {clean_c})"
                redundancy_warnings.append(msg)

        # Si faltan datos críticos, devolvemos False
        if missing:
            if len(missing) > 2:
                return False, f"ALERTA CRÍTICA: Datos perdidos: {', '.join(missing[:5])}..."
            return False, f"ALERTA: Faltan datos específicos: {', '.join(missing)}"

        # Si todo está bien pero hubo redundancias, devolvemos True pero con mensaje especial
        if redundancy_warnings:
            return True, f"[REDUNDANCIA] {', '.join(redundancy_warnings)}"

        return True, "OK"

    # --- MEJORA 2: Skeleton Diff (Responde a 'comparación tipo diff') ---
    def safety_structural_check(self, original, cleaned):
        """
        1. Intenta comparación estricta (Diff secuencial).
        2. Si falla (posible tabla reordenada), intenta comparación de conjuntos (Jaccard).
        """
        def clean_to_list(s):
            """
            Convierte un texto en una lista de palabras alfanuméricas en minúsculas,
            usada para la comparación por conjunto de palabras (índice de Jaccard).
            """
            # Convertir a lista de palabras alfanuméricas minúsculas
            return [w for w in re.split(r'\W+', s.lower()) if w.strip()]

        # 1. Chequeo Secuencial (Tu código actual)
        skel_orig = "".join(c for c in original if c.isalnum())
        skel_clean = "".join(c for c in cleaned if c.isalnum())

        matcher = difflib.SequenceMatcher(None, skel_orig, skel_clean)
        ratio_seq = matcher.ratio()

        if ratio_seq >= 0.90: # Bajamos un poco el umbral estricto
            return True, "OK"

        # 2. FALLBACK: Chequeo de "Bolsa de Palabras" (Para tablas reordenadas)
        # Si el ratio secuencial es bajo, verificamos si las palabras son las mismas aunque en otro orden.
        words_orig = set(clean_to_list(original))
        words_clean = set(clean_to_list(cleaned))

        # Intersección sobre Unión (Jaccard Index) o Cobertura
        common = words_orig.intersection(words_clean)
        coverage = len(common) / len(words_orig) if len(words_orig) > 0 else 0

        # Si recuperamos el 85% de las palabras únicas, damos por bueno el reordenamiento
        if coverage > 0.90:
            return True, f"OK"

        # Si falla también esto, reportamos las palabras clave perdidas
        return False, f"ALERTA DIFF: El contenido textual ha variado demasiado (Similitud: {ratio_seq:.2%})"

    def call_gemini_cleaning(self, text, filename, part_num, total_parts, previous_tail):
        """
        Envía un fragmento de texto a Gemini para su limpieza/formateo a
        Markdown, validando la respuesta con las comprobaciones de seguridad
        (longitud, semántica y estructura) y reintentando hasta 5 veces -
        escalando de gemini-2.5-flash a gemini-2.5-pro y, en los últimos
        intentos, a un prompt de rescate- antes de devolver el texto original
        sin procesar como último recurso.
        """
        prompt_standard = f"""
        Actúa como un motor OCR de corrección post-proceso.
        Tarea: Formatear a Markdown.
        Restricción MÁXIMA: Fidelidad 100% al contenido literal.

        Ejemplo:
        Input: "Paciente \ncon   \nHTA."
        Output: "Paciente con HTA."

        Contexto previo (para continuidad, NO REPETIR): "...{previous_tail}"

        TEXTO A PROCESAR:
        {text}
        """

        prompt_rescue = f"""
        URGENTE: PRESERVAR CONTENIDO.
        En intentos anteriores has borrado información.

        - MANTÉN todos los nombres, cantidades y cifras ÍNTEGROS.
        - Tu prioridad es la INTEGRIDAD, no la estética.
        - SI HAY DUDA ENTRE FORMATEAR Y CONSERVAR, CONSERVA.

        Contexto previo (para continuidad, NO REPETIR): "...{previous_tail}"

        TEXTO:
        {text}
        """

        max_retries = 5
        for attempt in range(max_retries):
            # Lógica de cambio de estrategia
            if attempt < 2:
                current_prompt = prompt_standard
                model = model_flash
            else:
                model = model_pro
                if attempt < 4:
                    current_prompt = prompt_standard
                else:
                    current_prompt = prompt_rescue
            try:
                response = model.generate_content(current_prompt)
                # --- LLAMADA A TU FUNCIÓN DE SEGURIDAD ---
                len_ok, len_msg = self.safety_length_check(text, response.text)
                sem_ok, sem_msg = self.safety_semantic_check(text, response.text)
                # 3. Check Estructural (Diff Skeleton) - NUEVO
                # Este verifica que no se haya inventado palabras o borrado frases enteras
                diff_ok, diff_msg = self.safety_structural_check(text, response.text)

                if not len_ok or not sem_ok or not diff_ok:
                    if not len_ok:
                        reason = len_msg
                    elif not sem_ok:
                        reason = sem_msg
                    else:
                        reason = diff_msg

                    print(f"{reason} en {filename} (Parte {part_num}). Reintentando {attempt+1}/{max_retries}...")
                    time.sleep(1)
                    continue

                # --- LÓGICA DE REGISTRO DE REDUNDANCIAS ---
                # Si sem_ok es True, pero el mensaje empieza por [REDUNDANCIA], lo guardamos.
                if sem_ok and "[REDUNDANCIA]" in sem_msg:
                    log_entry = f"DOC: {filename} | PARTE: {part_num}/{total_parts} | DATOS: {sem_msg.replace('[REDUNDANCIA] ', '')}"
                    self.redundancy_log.append(log_entry)
                    # Opcional: imprimir en consola también
                    print(f"  -> AVISO: {log_entry}")

                return response.text
            except Exception as e:
                if "429" in str(e):
                    wait = 5 + random.uniform(0, 5)
                    print(f"Rate limit alcanzado. Reintentando en {wait:.1f}s...")
                    time.sleep(wait)
                else:
                    print(f"\nError con Gemini en {filename} (Parte {part_num}): {e}")
                    time.sleep(2)
                    continue

        print(f"FALLO FINAL en {filename} (Parte {part_num}). Devolviendo texto ORIGINAL sucio para revisión manual.")
        # --- AQUÍ GUARDAMOS EL TEXTO CRUDO PARA EL TXT ---
        error_info = f"=== DOCUMENTO: {filename} | PARTE: {part_num}/{total_parts} ===\n\n{text}\n"
        self.error_log.append(error_info)
        # En call_gemini_cleaning (cuando falla todo):
        error_obj = {
            "filename": filename,
            "part_num": part_num,
            "total_parts": total_parts,
            "raw_text": text  # <--- Guardamos el texto EXACTO
        }
        self.error_log_json.append(error_obj)
        # -------------------------------------------------
        return text

    def process_json(self, json_data):
        """
        Procesa todos los documentos del JSON de entrada: une el texto de sus
        páginas, aplica la limpieza mecánica, lo divide en fragmentos (chunks)
        respetando los puntos y aparte, limpia cada fragmento con Gemini
        manteniendo continuidad entre fragmentos, y construye el documento
        final en formato page_content/metadata listo para el chunking.
        """
        processed_docs = []
        print(f"Procesando {len(json_data)} documentos con corte inteligente...")

        for entry in tqdm(json_data):
            filename = entry.get('file', 'unknown')

            # 1. Unir todo
            sorted_pages = sorted(entry.get('pages', []), key=lambda x: x.get('page', 0))
            all_text_list = [p.get('text', '') for p in sorted_pages if p.get('text')]
            full_raw_text = "\n".join(all_text_list)

            if len(full_raw_text) < 50: continue

            # 2. Limpieza mecánica
            step1 = self.clean_text_surface(full_raw_text)
            step2 = self.repair_broken_content(step1)

            # 3. Troceado Inteligente (Puntos y aparte)
            chunks = self.split_text_smartly(step2, chunk_size=5750)

            doc_markdown_parts = []

            # 4. Procesar con Gemini
            for i, chunk in enumerate(chunks):
                previous_tail = doc_markdown_parts[-1][-450:] if doc_markdown_parts else ""
                cleaned_chunk = self.call_gemini_cleaning(chunk, filename, i+1, len(chunks), previous_tail)
                doc_markdown_parts.append(cleaned_chunk)
                self.global_throttle()

            # 5. Unir y guardar
            final_text_complete = "\n\n".join(doc_markdown_parts)

            doc = {
                "page_content": final_text_complete,
                "metadata": {
                    "source": filename,
                    "domain": "protocolo_clinico_murcia",
                    "type": "text_markdown"
                }
            }
            processed_docs.append(doc)

            print(f" -> Documento '{filename}' terminado.")
            time.sleep(0.5)

        return processed_docs

# --- 3. EJECUCIÓN ---
def main():
    """
    Punto de entrada del script: carga el JSON de la Fase 1 ya limpiado, lo
    procesa con MedicalProtocolProcessor y guarda el corpus final en JSON,
    una versión en texto plano para revisión visual, y los reportes de
    incidencias/errores críticos detectados durante el proceso.
    """
    INPUT_FILE = '8_final_cleaned.json'
    OUTPUT_JSON = 'corpus_rag_final_gemini.json'
    OUTPUT_TXT = 'corpus_rag_visual_revisar.txt'
    REPORT_FILE = 'REPORTE_INCIDENCIAS_MANUAL.txt'
    ERROR_FILE = 'ERRORES_TEXTO_CRUDO.txt'          # Fallos críticos (textos para arreglar a mano)
    ERROR_FILE_json = 'ERRORES_TEXTO_CRUDO.json'          # Fallos críticos (textos para arreglar a mano)

    print(f"Cargando {INPUT_FILE}...")
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print("Error: No encuentro el archivo JSON de entrada.")
        return

    processor = MedicalProtocolProcessor()
    rag_docs = processor.process_json(data)

    print(f"Guardando JSON en {OUTPUT_JSON}...")
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(rag_docs, f, ensure_ascii=False, indent=4)

    print(f"Guardando TXT visual en {OUTPUT_TXT}...")
    with open(OUTPUT_TXT, 'w', encoding='utf-8') as f:
        f.write("=== REVISIÓN DE DOCUMENTOS (CORTE INTELIGENTE) ===\n\n")
        for i, doc in enumerate(rag_docs, 1):
            f.write(f"{'='*60}\n")
            f.write(f"DOC {i}: {doc['metadata']['source']}\n")
            f.write(f"{'='*60}\n\n")
            f.write(doc['page_content'])
            f.write("\n\n" + ("-"*40) + "\n\n")

    # --- GUARDAR REPORTE DE INCIDENCIAS ---
    print(f"\nGenerando reporte de revisión manual en {REPORT_FILE}...")
    with open(REPORT_FILE, 'w', encoding='utf-8') as f:
        f.write("=== REPORTE DE INCIDENCIAS Y REDUNDANCIAS ===\n")
        f.write("Revisar manualmente estos casos para asegurar que no se ha perdido info.\n\n")
        if processor.redundancy_log:
            for log in processor.redundancy_log:
                f.write(log + "\n")

    # --- 2. GUARDAR REPORTE DE FALLOS CRUDOS ---
    print(f"Generando archivo de errores crudos en {ERROR_FILE} y {ERROR_FILE_json}...")

    # Abrimos el archivo TXT para escribir, haya errores o no
    with open(ERROR_FILE, 'w', encoding='utf-8') as f_txt:
        if processor.error_log:
            f_txt.write("=== PARTES QUE FALLARON Y SE DEVOLVIERON EN CRUDO ===\n")
            for err in processor.error_log:
                f_txt.write(err)
                f_txt.write("-" * 80 + "\n\n")
        else:
            f_txt.write("Todos los bloques se procesaron correctamente sin fallos críticos.\n")

    # Guardamos el JSON solo si hay errores (o un array vacío si prefieres consistencia)
    if processor.error_log_json:
        with open(ERROR_FILE_json, 'w', encoding='utf-8') as f_json:
            json.dump(processor.error_log_json, f_json, ensure_ascii=False, indent=4)
    else:
        # Opcional: Crear un JSON vacío para indicar éxito explícito
        with open(ERROR_FILE_json, 'w', encoding='utf-8') as f_json:
            json.dump([], f_json)


    print("\n¡PROCESO COMPLETADO!.")

if __name__ == "__main__":
    main()

Cargando 8_final_cleaned.json...
Procesando 20 documentos con corte inteligente...


  0%|          | 0/20 [00:00<?, ?it/s]

ALERTA: El texto se ha reducido demasiado (21.96%). Posible pérdida de info. en 1181_Guía hospitalaria de terapéutica antibiótica en adultos.pdf (Parte 8). Reintentando 1/5...
ALERTA: El texto se ha reducido demasiado (21.96%). Posible pérdida de info. en 1181_Guía hospitalaria de terapéutica antibiótica en adultos.pdf (Parte 8). Reintentando 2/5...
ALERTA: El texto se ha reducido demasiado (10.52%). Posible pérdida de info. en 1181_Guía hospitalaria de terapéutica antibiótica en adultos.pdf (Parte 9). Reintentando 1/5...
ALERTA: El texto se ha reducido demasiado (10.52%). Posible pérdida de info. en 1181_Guía hospitalaria de terapéutica antibiótica en adultos.pdf (Parte 9). Reintentando 2/5...
  -> AVISO: DOC: 1181_Guía hospitalaria de terapéutica antibiótica en adultos.pdf | PARTE: 9/34 | DATOS: 60ml (Input: 6 -> Output: 2), 30ml (Input: 3 -> Output: 1), 10ml (Input: 3 -> Output: 1)
ALERTA CRÍTICA: Datos perdidos: 75mg (DESAPARECIDO), 2g (DESAPARECIDO), 500mg (DESAPARECIDO)... en 118

  5%|▌         | 1/20 [20:39<6:32:34, 1239.69s/it]

ALERTA DIFF: El contenido textual ha variado demasiado (Similitud: 0.00%) en 4084_Protocolo regional de prevención y detección de violencia en la mujer mayor de 65 años.pdf (Parte 8). Reintentando 1/5...
ALERTA DIFF: El contenido textual ha variado demasiado (Similitud: 89.39%) en 4084_Protocolo regional de prevención y detección de violencia en la mujer mayor de 65 años.pdf (Parte 8). Reintentando 2/5...
 -> Documento '4084_Protocolo regional de prevención y detección de violencia en la mujer mayor de 65 años.pdf' terminado.


...

 -> Documento '11864_Procedimiento para el cumplimiento de la objeción de conciencia de los profesionales sanitarios implicados en la prestación de ayuda a morir.pdf' terminado.


 95%|█████████▌| 19/20 [1:05:41<01:40, 100.49s/it]

 -> Documento '10604_Protocolo de uso de Nivolumab en condiciones distintas a las de la ficha técnica (dosificación por kg de peso) en las indicaciones aprobadas.pdf' terminado.


100%|██████████| 20/20 [1:05:56<00:00, 197.83s/it]

Guardando JSON en corpus_rag_final_gemini.json...
Guardando TXT visual en corpus_rag_visual_revisar.txt...

Generando reporte de revisión manual en REPORTE_INCIDENCIAS_MANUAL.txt...
Generando archivo de errores crudos en ERRORES_TEXTO_CRUDO.txt y ERRORES_TEXTO_CRUDO.json...

¡PROCESO COMPLETADO!.


In [ ]:
from google.colab import files
import os
import zipfile

# Lista de tus archivos de salida
filenames = [
    'corpus_rag_final_gemini.json',
    'corpus_rag_visual_revisar.txt',
    'REPORTE_INCIDENCIAS_MANUAL.txt',
    'ERRORES_TEXTO_CRUDO.txt',
    'ERRORES_TEXTO_CRUDO.json'
]

zip_name = 'resultados_procesamiento_rag.zip'

print(f"Comprimiendo archivos en {zip_name}...")

with zipfile.ZipFile(zip_name, 'w') as zipf:
    found_any = False
    for file in filenames:
        if os.path.exists(file):
            print(f"  -> Añadiendo: {file}")
            zipf.write(file)
            found_any = True
        else:
            print(f"  Warning: No se encontró {file} (quizás no se generó)")

if found_any:
    print("¡Descargando ZIP!")
    files.download(zip_name)
else:
    print("No se encontraron archivos para descargar.")

Comprimiendo archivos en resultados_procesamiento_rag.zip...
  -> Añadiendo: corpus_rag_final_gemini.json
  -> Añadiendo: corpus_rag_visual_revisar.txt
  -> Añadiendo: REPORTE_INCIDENCIAS_MANUAL.txt
  -> Añadiendo: ERRORES_TEXTO_CRUDO.txt
  -> Añadiendo: ERRORES_TEXTO_CRUDO.json
¡Descargando ZIP!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import json
import os
import statistics

def analizar_granularidad():
    """
    Calcula y muestra estadísticas de granularidad del corpus final (número
    de bloques semánticos por documento y su longitud media/mediana/máxima/
    mínima), tomando como bloque cada fragmento de texto separado por doble
    salto de línea en el Markdown.
    """
    INPUT_FILE = 'corpus_rag_final_gemini.json'

    print(f"--- ANÁLISIS DE GRANULARIDAD ---")

    if not os.path.exists(INPUT_FILE):
        print(f"Error: No encuentro el archivo {INPUT_FILE}")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)

    total_docs = len(data)
    all_blocks_lengths = []
    total_blocks_count = 0

    print(f"Analizando {total_docs} documentos...\n")

    for doc in data:
        content = doc['page_content']

        # DEFINICIÓN DE BLOQUE SEMÁNTICO:
        # En Markdown, los bloques (párrafos, ítems de lista, tablas)
        # suelen estar separados por doble salto de línea (\n\n).
        # Esto es más preciso que contar líneas sueltas.
        bloques = [b.strip() for b in content.split('\n\n') if b.strip()]

        num_bloques = len(bloques)
        total_blocks_count += num_bloques

        # Guardamos la longitud de cada bloque para la media
        for b in bloques:
            all_blocks_lengths.append(len(b))

    # --- CÁLCULOS ---
    avg_blocks_per_doc = total_blocks_count / total_docs if total_docs > 0 else 0
    avg_char_per_block = statistics.mean(all_blocks_lengths) if all_blocks_lengths else 0
    median_char_per_block = statistics.median(all_blocks_lengths) if all_blocks_lengths else 0

    max_char_per_block = max(all_blocks_lengths) if all_blocks_lengths else 0
    min_char_per_block = min(all_blocks_lengths) if all_blocks_lengths else 0

    # --- REPORTE FINAL ---
    print("="*60)
    print("RESUMEN DE MÉTRICAS DEL CORPUS")
    print("="*60)
    print(f"1. Documentos Originales (PDFs/Archivos):  {total_docs}")
    print(f"2. Total de Bloques Semánticos (Párrafos): {total_blocks_count}")
    print("-" * 60)
    print(f"3. Promedio de bloques por documento:      {avg_blocks_per_doc:.1f}")
    print(f"4. Longitud media del bloque (caracteres): {avg_char_per_block:.0f}")
    print(f"5. Longitud mediana del bloque:            {median_char_per_block:.0f}")
    print(f"6. Longitud MÁXIMA del bloque:             {max_char_per_block}")
    print(f"7. Longitud MÍNIMA del bloque:             {min_char_per_block}")
    print("="*60)

if __name__ == "__main__":
    analizar_granularidad()

--- ANÁLISIS DE GRANULARIDAD ---
Analizando 20 documentos...

RESUMEN DE MÉTRICAS DEL CORPUS
1. Documentos Originales (PDFs/Archivos):  20
2. Total de Bloques Semánticos (Párrafos): 1862
------------------------------------------------------------
3. Promedio de bloques por documento:      93.1
4. Longitud media del bloque (caracteres): 602
5. Longitud mediana del bloque:            349
6. Longitud MÁXIMA del bloque:             5775
7. Longitud MÍNIMA del bloque:             2
